# Phase 3: Pathogen Gate and Domain Questions

First Jev scores whether each species–abstract pair provides human disease evidence. Pairs with `p_yes` at or above `confidence_threshold` (default 0.75) then receive all 73 V1 domain questions in one Jev Decisions API request per species–abstract pair. Scores are cached and resumable.

Run the configuration cell in notebook 00 first to change Phase 3 settings. Set `RUN_PHASE_OVERRIDE = True` (or `run_phase: true` in `configs/notebook_parameters.yaml`) to run both Jev stages.


In [7]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v2").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v2 import notebook_utils as nbutils
from pathogen_atlas_v2.common.io import ensure_parent, resolve_path
from pathogen_atlas_v2.llm.pathogen_evidence import apply_confidence_threshold, phase3_preflight, run_phase3
from pathogen_atlas_v2.llm.domain_expert import (
    apply_domain_question_threshold,
    domain_expert_preflight,
    load_current_domain_expert_probabilities,
    run_domain_expert_phase,
)

NOTEBOOK_CONFIG_PATH = "configs/notebook_parameters.yaml"
PARAMS = nbutils.notebook_phase_parameters("phase3_abstract_decisions", NOTEBOOK_CONFIG_PATH)
RUN_PHASE_OVERRIDE = None  # Set True to run now, False to skip, or None to use YAML.
RUN_PHASE = bool(PARAMS.get("run_phase", False)) if RUN_PHASE_OVERRIDE is None else bool(RUN_PHASE_OVERRIDE)
RUN_ID = str(PARAMS.get("run_id", "jev-human-pathogen-evidence-v1"))
DECISION_PATH = resolve_path(PARAMS.get("decision_path", "data/03_decisions/parsed/pathogen_evidence_jev.parquet"))
ACCEPTED_DECISIONS_PATH = resolve_path(PARAMS.get("accepted_decisions_path", "data/03_decisions/parsed/pathogen_evidence_jev_accepted.parquet"))
CONFIDENCE_THRESHOLD = float(PARAMS.get("confidence_threshold", 0.75))
QUESTION_CONFIDENCE_THRESHOLD = float(PARAMS.get("question_confidence_threshold", 0.75))
EXPERT_ANSWERS_PATH = resolve_path(PARAMS.get("expert_answers_path", "data/03_decisions/parsed/domain_expert_answers.parquet"))

## Preflight

This reads Phase 2 release tables and the local decision ledger. It does not call Jev. Check the pending pair count, selected model, question version, and maximum call count before enabling the run.

In [8]:
PREFLIGHT = phase3_preflight(PARAMS)
PREFLIGHT["confidence_threshold"] = CONFIDENCE_THRESHOLD
display(pd.DataFrame([PREFLIGHT]))

,status,provider,model,question_id,question_version,run_id,selected_species_pmid_pairs,species_count,missing_abstract_pairs,already_completed_for_run,pending_pairs,max_calls,scheduled_api_calls,decision_path,raw_response_path,confidence_threshold
0,ready,openrouter_jev,typesafe/jev-1.14,human_pathogen_evidence,human-pathogen-disease-attribution-v1,jev-human-pathogen-evidence-v1,161471,5969,6,0,161471,25,25,/Users/f.costa/Documents/Projects/Repositories...,/Users/f.costa/Documents/Projects/Repositories...,0.75


## OpenRouter credentials

Jev needs `OPENROUTER_API_KEY` in the current notebook kernel. If it is not already set, the next cell asks for it using hidden input. The key is held only in this kernel session and is not written to the notebook or YAML config.

In [ ]:
import os
from getpass import getpass

if RUN_PHASE:
    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip()
    if not api_key:
        api_key = getpass("OpenRouter API key (input hidden): ").strip()
    if not api_key:
        raise RuntimeError("No OpenRouter API key was provided. Enter the key or set OPENROUTER_API_KEY in this kernel before running Jev.")
    os.environ["OPENROUTER_API_KEY"] = api_key
    del api_key
    print("OpenRouter key is available to this kernel session.")
else:
    print("RUN_PHASE is false; skipping OpenRouter credential setup.")

## Run Jev decisions

The runner skips successful rows whose run ID, input hash, model, and question version match. Failed responses remain eligible for a later retry. Keep `max_calls` at 25 for a pilot; set it to `None` only when you intend to process all pending pairs.

In [ ]:
if RUN_PHASE:
    if PREFLIGHT.get("status") != "ready":
        raise RuntimeError(f"Phase 3 is not ready: {PREFLIGHT}")
    PHASE3_RESULT = run_phase3(PARAMS, log=True)
else:
    PHASE3_RESULT = {
        "status": "not_run",
        "reason": "Set run_phase: true in the Phase 3 notebook parameters to call Jev.",
        "preflight": PREFLIGHT,
    }
display(PHASE3_RESULT)

## Apply the confidence threshold

The notebook labels a completed decision as accepted when `p_yes >= confidence_threshold`. The default is 0.75 and can be changed in `configs/notebook_parameters.yaml`. Failed or missing decisions remain unaccepted. The raw probability ledger is unchanged; accepted rows are written to a separate Parquet file.


In [ ]:
if DECISION_PATH.exists():
    DECISIONS = pd.read_parquet(DECISION_PATH)
    if "run_id" in DECISIONS.columns:
        CURRENT_RUN = DECISIONS.loc[DECISIONS["run_id"].astype(str).eq(RUN_ID)].copy()
    else:
        CURRENT_RUN = DECISIONS.iloc[0:0].copy()
    CURRENT_RUN = apply_confidence_threshold(CURRENT_RUN, CONFIDENCE_THRESHOLD)
    ACCEPTED_DECISIONS = CURRENT_RUN.loc[CURRENT_RUN["accepted"].fillna(False)].copy()
    ensure_parent(ACCEPTED_DECISIONS_PATH)
    ACCEPTED_DECISIONS.to_parquet(ACCEPTED_DECISIONS_PATH, index=False)
    if CURRENT_RUN.empty:
        print(f"No saved decisions for run {RUN_ID!r} yet.")
    else:
        display(CURRENT_RUN["status"].value_counts(dropna=False).rename_axis("status").to_frame("rows"))
        display(CURRENT_RUN["accepted"].value_counts(dropna=False).rename_axis("accepted at threshold").to_frame("rows"))
        display(ACCEPTED_DECISIONS.head(20))
        print(f"Accepted {len(ACCEPTED_DECISIONS)} of {len(CURRENT_RUN)} decisions at p_yes >= {CONFIDENCE_THRESHOLD:.2f}.")
    print(f"Accepted decisions saved to: {ACCEPTED_DECISIONS_PATH}")
else:
    print(f"Decision ledger does not exist yet: {DECISION_PATH}")


## Domain expert questions

The saved pathogen-gate ledger is applied first. Only unchanged species–abstract pairs from the current gate run with completed `p_yes >= confidence_threshold` enter this stage. Jev receives the title and abstract once and answers all 73 versioned V1 yes/no questions together in one typed request.

Each species–abstract request is cached as raw JSONL; normalized question probabilities are checkpointed to Parquet. `expert_max_calls` limits species–abstract requests per run, not individual questions.


In [ ]:
EXPERT_PREFLIGHT = domain_expert_preflight(PARAMS)
display(pd.DataFrame([EXPERT_PREFLIGHT]))


## Run and inspect domain questions

When `RUN_PHASE` is enabled, this runs the expert-question stage after the pathogen gate. Each returned probability is converted to a yes/no answer using `question_confidence_threshold` (default 0.75); failed questions remain blank.


In [ ]:
if RUN_PHASE:
    if EXPERT_PREFLIGHT.get("status") != "ready":
        raise RuntimeError(f"Domain expert questions are not ready: {EXPERT_PREFLIGHT}")
    EXPERT_RESULT = run_domain_expert_phase(PARAMS, log=True)
else:
    EXPERT_RESULT = {"status": "not_run", "preflight": EXPERT_PREFLIGHT}
display(EXPERT_RESULT)

EXPERT_PROBABILITIES = load_current_domain_expert_probabilities(PARAMS)
EXPERT_ANSWERS = apply_domain_question_threshold(
    EXPERT_PROBABILITIES,
    QUESTION_CONFIDENCE_THRESHOLD,
)
ensure_parent(EXPERT_ANSWERS_PATH)
EXPERT_ANSWERS.to_parquet(EXPERT_ANSWERS_PATH, index=False)
if EXPERT_ANSWERS.empty:
    print("No saved domain-question answers for the current gate-passing abstracts yet.")
else:
    display(EXPERT_ANSWERS.groupby(["category", "answer_yes"], dropna=False).size().rename("rows").to_frame())
    display(EXPERT_ANSWERS.head(40))
print(f"Domain-question answers saved to: {EXPERT_ANSWERS_PATH}")
